In [154]:
# Biblioteker
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, RandomEffects
from scipy.stats import chi2
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

In [155]:
# Indlæs data
GDP = pd.read_csv("Data/GDP.csv", skiprows=4)
ODR = pd.read_csv("Data/age-dependency-ratio-old.csv")
TFP = pd.read_csv("Data/total-factor-productivity.csv")
HC_DATA = pd.read_excel("Data/pwt110.xlsx", sheet_name="Data")
HC = HC_DATA[['country', 'year', 'hc', 'ctfp', 'rgdpe', 'pop']]
GEO = pd.read_excel("Data/geo_cepii.xls")
OPN = pd.read_csv("Data/Oppenness.csv", skiprows=4)
URB = pd.read_csv("Data/Urbanization.csv", skiprows=4)

# Sammensæt data
Formålet er at konstruere et tværsnitsdatasæt med ét datapunkt pr. land, hvor vi forklarer TFP-vækst fra 2002 til 2020 med initial ODR og kontroller fra 2002

In [156]:
# Relevante variabler fra HC_DATA
HC = HC_DATA[['country','year','hc','ctfp','rgdpe','pop']].copy()

# Relevante geografiske variable
GEO = GEO[['country', 'lat', 'continent', 'landlocked', 'colonizer1']].copy()

# Afstand til ækvator
GEO['AbsLat'] = GEO['lat'].abs()

# Tidligere koloni
GEO['FormerColony'] = (GEO['colonizer1'] != '.').astype(int)

# Behold kun relevante variable
GEO = GEO[['country', 'AbsLat', 'continent', 'landlocked', 'FormerColony']]

# Trade openness i 2002
opn2002 = OPN[['Country Name', '2002']].copy()
opn2002.columns = ['country', 'TradeOpen2002']

# Urbanisering i 2002
URB = URB[['Country Name', '2002']].copy()
URB.columns = ['country', 'Urban2002']
URB['Urban2002'] = pd.to_numeric(URB['Urban2002'], errors='coerce')

# BNP per capita
HC['gdp_pc'] = HC['rgdpe'] / HC['pop']

# Initiale variable i 2002
base2002 = HC[HC['year'] == 2002][['country','hc','ctfp','gdp_pc']].copy()
base2002.columns = ['country','HC2002','TFP2002','GDPpc2002']

# TFP i 2020
tfp2020 = HC[HC['year'] == 2020][['country','ctfp']].copy()
tfp2020.columns = ['country','TFP2020']

# Long-difference datasæt
Tvaersnit = pd.merge(base2002, tfp2020, on='country', how='inner')

# GrowthTFP
Tvaersnit['GrowthTFP'] = np.log(Tvaersnit['TFP2020']) - np.log(Tvaersnit['TFP2002'])

# ODR i 2002
odr2002 = ODR[ODR['Year'] == 2002][[
    'Entity',
    'Age dependency ratio, old (% of working-age population)'
]].copy()
odr2002.columns = ['country', 'ODR2002']

# Samlet tværsnit
Tvaersnit = pd.merge(Tvaersnit, odr2002, on='country', how='inner')
Tvaersnit = pd.merge(Tvaersnit, GEO, on='country', how='left')
Tvaersnit = pd.merge(Tvaersnit, opn2002, on='country', how='left')
Tvaersnit = pd.merge(Tvaersnit, URB, on='country', how='left')

# Fjern manglende observationer
Tvaersnit = Tvaersnit.dropna()

# Fjern dubletter
Tvaersnit = Tvaersnit.drop_duplicates(subset='country', keep='first')

In [157]:
Tvaersnit['FormerColony'].value_counts()

FormerColony
1.0    73
0.0    18
Name: count, dtype: int64

# Deskriptiv statistik

In [158]:
print(Tvaersnit.describe())

          HC2002    TFP2002     GDPpc2002    TFP2020  GrowthTFP    ODR2002  \
count  91.000000  91.000000     91.000000  91.000000  91.000000  91.000000   
mean    2.480295   0.677952  19081.668378   0.639423  -0.019644  12.744400   
std     0.653893   0.290769  18097.596199   0.210196   0.273470   7.868767   
min     1.088122   0.167584   1048.418252   0.200132  -0.515328   1.583193   
25%     2.065554   0.400263   4776.914256   0.475570  -0.226324   6.044639   
50%     2.538953   0.652943  11654.836917   0.622159  -0.050273   8.787122   
75%     2.964713   0.874805  34026.379648   0.813353   0.170682  20.635919   
max     3.583555   1.394810  84592.225755   1.078187   0.942964  28.059470   

          AbsLat  landlocked  FormerColony  TradeOpen2002   Urban2002  
count  91.000000   91.000000     91.000000      91.000000   91.000000  
mean   29.614575    0.164835      0.802198      79.536134   61.162942  
std    17.889135    0.373087      0.400549      47.808033   21.438423  
min     0

# ODR + geo variabler

Growth_TFP = α + β1 ODR2002 + β2 TradeOpen2002 + β3 Urban2002 + β4 landlocked + ε

In [159]:
# Klargør datasæt
df_model = Tvaersnit[[
    'country',
    'GrowthTFP',
    'ODR2002',
    'Urban2002',
    'AbsLat',
    'TradeOpen2002',
    'landlocked',
    'FormerColony'
]].dropna().copy()

# Antal lande og observationer
antal_lande = df_model['country'].nunique()
antal_observationer = df_model.shape[0]

# Automatisk print
print("=" * 80)
print("MODEL:")
print("GrowthTFP = α + β1 ODR2002 + β2 TradeOpen2002 + β3 Urban2002 + β4 landlocked + ε")
print(f"Antal lande: {antal_lande}")
print(f"Antal observationer: {antal_observationer}")
print("=" * 80)

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): Kun ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): + TradeOpen2002
X2 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + log_area
X3 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + landlocked
X4 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + landlocked
X5 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'AbsLat']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')

# Model (6): + landlocked
X6 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'FormerColony']])
model6 = sm.OLS(y, X6).fit(cov_type='HC1')

# Tabel
results = summary_col(
    [model1, model2, model3, model4, model5, model6],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)', '(6)'],
    regressor_order=['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked', 'AbsLat', 'FormerColony'],
    drop_omitted=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)

MODEL:
GrowthTFP = α + β1 ODR2002 + β2 TradeOpen2002 + β3 Urban2002 + β4 landlocked + ε
Antal lande: 91
Antal observationer: 91

                  (1)        (2)        (3)        (4)       (5)       (6)   
-----------------------------------------------------------------------------
ODR2002        -0.0091*** -0.0091*** -0.0104*** -0.0104*** -0.0087  -0.0099**
               (0.0031)   (0.0032)   (0.0039)   (0.0039)   (0.0058) (0.0045) 
TradeOpen2002             -0.0003    -0.0004    -0.0004    -0.0004  -0.0004  
                          (0.0004)   (0.0004)   (0.0004)   (0.0004) (0.0004) 
Urban2002                            0.0011     0.0009     0.0012   0.0011   
                                     (0.0015)   (0.0016)   (0.0016) (0.0015) 
landlocked                                      -0.0286                      
                                                (0.0925)                     
AbsLat                                                     -0.0010           
             

# HC + geo variabler

Growth_TFP = α + β1 ODR2002 + β2 HC2002 + β3 TradeOpen2002 + β4 Urban2002 + β5 landlocked + ε

In [160]:
# Klargør datasæt
df_model = Tvaersnit[[
    'country',
    'GrowthTFP',
    'ODR2002',
    'HC2002',
    'TradeOpen2002',
    'Urban2002',
    'landlocked'
]].dropna().copy()

# Antal lande og observationer
antal_lande = df_model['country'].nunique()
antal_observationer = df_model.shape[0]

# Automatisk print
print("=" * 95)
print("MODEL:")
print("GrowthTFP = α + β1 ODR2002 + β2 HC2002 + β3 TradeOpen2002 + β4 Urban2002 + β5 landlocked + ε")
print(f"Antal lande: {antal_lande}")
print(f"Antal observationer: {antal_observationer}")
print("=" * 95)

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): const + ODR + HC2002
X2 = sm.add_constant(df_model[['ODR2002', 'HC2002']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + TradeOpen2002
X3 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'TradeOpen2002']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + Urban2002
X4 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'TradeOpen2002', 'Urban2002']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + landlocked
X5 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'TradeOpen2002', 'Urban2002', 'landlocked']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')

# Samlet tabel
results = summary_col(
    [model1, model2, model3, model4, model5],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)'],
    regressor_order=['ODR2002', 'HC2002', 'TradeOpen2002', 'Urban2002', 'landlocked'],
    drop_omitted=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)

MODEL:
GrowthTFP = α + β1 ODR2002 + β2 HC2002 + β3 TradeOpen2002 + β4 Urban2002 + β5 landlocked + ε
Antal lande: 91
Antal observationer: 91

                  (1)        (2)        (3)        (4)        (5)    
---------------------------------------------------------------------
ODR2002        -0.0091*** -0.0127*** -0.0140*** -0.0141*** -0.0143***
               (0.0031)   (0.0047)   (0.0050)   (0.0050)   (0.0049)  
HC2002                    0.0570     0.0783     0.0698     0.0739    
                          (0.0718)   (0.0785)   (0.0861)   (0.0840)  
TradeOpen2002                        -0.0005    -0.0006    -0.0005   
                                     (0.0004)   (0.0004)   (0.0005)  
Urban2002                                       0.0005     0.0003    
                                                (0.0017)   (0.0017)  
landlocked                                                 -0.0376   
                                                           (0.0883)  
R-squared      0.06

# Log(GDP) + geo variabler 

Growth_TFP = α + β1 ODR2002 + β2 log(GDPpc2002) + β3 TradeOpen2002 + β4 Urban2002 + β5 landlocked + ε


In [161]:
# Klargør datasæt
df_model = Tvaersnit[[
    'country',
    'GrowthTFP',
    'ODR2002',
    'GDPpc2002',
    'TradeOpen2002',
    'Urban2002',
    'landlocked'
]].dropna().copy()

# Log-variable
df_model['log_GDPpc2002'] = np.log(df_model['GDPpc2002'])

# Antal lande og observationer
antal_lande = df_model['country'].nunique()
antal_observationer = df_model.shape[0]

# Automatisk print
print("=" * 100)
print("MODEL:")
print("GrowthTFP = α + β1 ODR2002 + β2 log(GDPpc2002) + β3 TradeOpen2002 + β4 Urban2002 + β5 landlocked + ε")
print(f"Antal lande: {antal_lande}")
print(f"Antal observationer: {antal_observationer}")
print("=" * 100)

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): const + ODR + log_GDPpc2002
X2 = sm.add_constant(df_model[['ODR2002', 'log_GDPpc2002']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + TradeOpen2002
X3 = sm.add_constant(df_model[['ODR2002', 'log_GDPpc2002', 'TradeOpen2002']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + Urban2002
X4 = sm.add_constant(df_model[['ODR2002', 'log_GDPpc2002', 'TradeOpen2002', 'Urban2002']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + landlocked
X5 = sm.add_constant(df_model[['ODR2002', 'log_GDPpc2002', 'TradeOpen2002', 'Urban2002', 'landlocked']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')

# Samlet tabel
results = summary_col(
    [model1, model2, model3, model4, model5],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)'],
    regressor_order=['ODR2002', 'log_GDPpc2002', 'TradeOpen2002', 'Urban2002', 'landlocked'],
    drop_omitted=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)

MODEL:
GrowthTFP = α + β1 ODR2002 + β2 log(GDPpc2002) + β3 TradeOpen2002 + β4 Urban2002 + β5 landlocked + ε
Antal lande: 91
Antal observationer: 91

                  (1)       (2)      (3)       (4)        (5)    
-----------------------------------------------------------------
ODR2002        -0.0091*** -0.0029  -0.0023  0.0005     0.0008    
               (0.0031)   (0.0041) (0.0043) (0.0043)   (0.0042)  
log_GDPpc2002             -0.0625* -0.0688* -0.1786*** -0.1824***
                          (0.0329) (0.0378) (0.0570)   (0.0566)  
TradeOpen2002                      0.0003   0.0004     0.0005    
                                   (0.0004) (0.0004)   (0.0005)  
Urban2002                                   0.0065***  0.0063*** 
                                            (0.0021)   (0.0022)  
landlocked                                             -0.0575   
                                                       (0.0861)  
R-squared      0.0693     0.1086   0.1103   0.2029     0.20

# Regioner + Geo variabler

GrowthTFP = α + β1 ODR2002 + β2 Europe + β3 Asia + β4 Africa + β5 LatinAmerica + β6 NorthAmerica + β7 Oceania + β8 MiddleEast + β9 TradeOpen2002 + β10 Urban2002 + β11 landlocked + ε


In [162]:
# Klargør datasæt
df_model = Tvaersnit[[
    'country',
    'GrowthTFP',
    'ODR2002',
    'GDPpc2002',
    'TradeOpen2002',
    'Urban2002',
    'landlocked'
]].dropna().copy()

# Log-variable
df_model['log_GDPpc2002'] = np.log(df_model['GDPpc2002'])

# Antal lande og observationer
antal_lande = df_model['country'].nunique()
antal_observationer = df_model.shape[0]

# Automatisk print
print("=" * 185)
print("MODEL:")
print("GrowthTFP = α + β1 ODR2002 + β2 Europe + β3 Asia + β4 Africa + β5 LatinAmerica + β6 NorthAmerica + β7 Oceania + β8 MiddleEast + β9 TradeOpen2002 + β10 Urban2002 + β11 landlocked + ε")
print(f"Antal lande: {antal_lande}")
print(f"Antal observationer: {antal_observationer}")
print("=" * 185)

# Regioner
europe = [
    'Albania','Austria','Belgium','Bulgaria','Croatia','Cyprus','Czechia','Denmark','Estonia','Finland','France','Germany',
    'Greece','Hungary','Iceland','Ireland','Italy','Latvia','Lithuania','Luxembourg','Malta','Netherlands','Norway',
    'Poland','Portugal','Romania','Serbia','Slovakia','Slovenia','Spain','Sweden','Switzerland','Ukraine','United Kingdom','Armenia']

asia = [
    'Bahrain','China','India','Indonesia','Iraq','Israel','Japan','Jordan','Kazakhstan','Kuwait','Kyrgyzstan','Malaysia',
    'Mongolia','Philippines','Qatar','Saudi Arabia','Singapore','Sri Lanka','Taiwan','Tajikistan','Thailand']

africa = [
    'Angola','Benin','Botswana','Burkina Faso','Burundi','Cameroon','Central African Republic','Egypt','Eswatini',
    'Gabon','Kenya','Lesotho','Mauritania','Mauritius','Morocco','Mozambique','Namibia','Niger','Nigeria',
    'Rwanda','Senegal','Sierra Leone','South Africa','Sudan','Togo','Tunisia','Zambia','Zimbabwe']

latin_america = [
    'Argentina','Barbados','Brazil','Chile','Colombia','Costa Rica','Dominican Republic','Ecuador','El Salvador',
    'Guatemala','Honduras','Jamaica','Mexico','Nicaragua','Panama','Paraguay','Peru','Trinidad and Tobago','Uruguay']

north_america = ['Canada', 'United States']

oceania = ['Australia', 'Fiji', 'New Zealand']

middle_east = ['Bahrain', 'Iraq', 'Israel', 'Jordan', 'Kuwait', 'Qatar', 'Saudi Arabia']

# Regionale dummies
df_model['Europe'] = df_model['country'].isin(europe).astype(int)
df_model['Asia'] = df_model['country'].isin(asia).astype(int)
df_model['Africa'] = df_model['country'].isin(africa).astype(int)
df_model['LatinAmerica'] = df_model['country'].isin(latin_america).astype(int)
df_model['NorthAmerica'] = df_model['country'].isin(north_america).astype(int)
df_model['Oceania'] = df_model['country'].isin(oceania).astype(int)
df_model['MiddleEast'] = df_model['country'].isin(middle_east).astype(int)

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): const + ODR + regionale dummies
X2 = sm.add_constant(df_model[['ODR2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + TradeOpen2002
X3 = sm.add_constant(df_model[['ODR2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast', 'TradeOpen2002']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + Urban2002
X4 = sm.add_constant(df_model[['ODR2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast',
                               'TradeOpen2002', 'Urban2002']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + landlocked
X5 = sm.add_constant(df_model[['ODR2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast',
                               'TradeOpen2002', 'Urban2002', 'landlocked']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')

results = summary_col(
    [model1, model2, model3, model4, model5],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)'],
    regressor_order=[
        'ODR2002', 'Europe', 'Asia', 'Africa',
        'LatinAmerica', 'NorthAmerica', 'Oceania',
        'MiddleEast', 'TradeOpen2002', 'Urban2002',
        'landlocked'
    ],
    drop_omitted=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)

MODEL:
GrowthTFP = α + β1 ODR2002 + β2 Europe + β3 Asia + β4 Africa + β5 LatinAmerica + β6 NorthAmerica + β7 Oceania + β8 MiddleEast + β9 TradeOpen2002 + β10 Urban2002 + β11 landlocked + ε
Antal lande: 91
Antal observationer: 91

                  (1)       (2)      (3)       (4)       (5)   
---------------------------------------------------------------
ODR2002        -0.0091*** -0.0078  -0.0096  -0.0173*  -0.0175* 
               (0.0031)   (0.0074) (0.0079) (0.0091)  (0.0091) 
Europe                    -0.0213  0.0102   0.0447    0.0485   
                          (0.0929) (0.1039) (0.1038)  (0.1017) 
Asia                      0.1090** 0.1308** 0.1684*** 0.1678***
                          (0.0525) (0.0557) (0.0564)  (0.0577) 
Africa                    -0.0032  -0.0164  -0.0038   -0.0015  
                          (0.0836) (0.0890) (0.0884)  (0.0914) 
LatinAmerica              -0.0038  -0.0094  -0.0766   -0.0766  
                          (0.0639) (0.0650) (0.0762)  (0.0774) 
No

# HC + log(GDP) + regionale dummies + geo variabler

In [163]:
# Klargør datasæt
df_model = Tvaersnit[[
    'country',
    'GrowthTFP',
    'HC2002',
    'ODR2002',
    'GDPpc2002',
    'TradeOpen2002',
    'Urban2002',
    'landlocked'
]].dropna().copy()

# Log-variable
df_model['log_GDPpc2002'] = np.log(df_model['GDPpc2002'])

# Antal lande og observationer
antal_lande = df_model['country'].nunique()
antal_observationer = df_model.shape[0]

# Automatisk print
print("=" * 220)
print("MODEL:")
print("GrowthTFP = α + β1 ODR2002 + β2 HC2002 + β3 log(GDPpc2002) + β4 Europe + β5 Asia + β6 Africa + β7 LatinAmerica + β8 NorthAmerica + β9 Oceania + β10 MiddleEast + β11 TradeOpen2002 + β12 Urban2002 + β13 landlocked + ε")
print(f"Antal lande: {antal_lande}")
print(f"Antal observationer: {antal_observationer}")
print("=" * 220)

# Regioner
europe = [
    'Albania','Austria','Belgium','Bulgaria','Croatia','Cyprus','Czechia','Denmark','Estonia','Finland','France','Germany',
    'Greece','Hungary','Iceland','Ireland','Italy','Latvia','Lithuania','Luxembourg','Malta','Netherlands','Norway',
    'Poland','Portugal','Romania','Serbia','Slovakia','Slovenia','Spain','Sweden','Switzerland','Ukraine','United Kingdom','Armenia']

asia = [
    'Bahrain','China','India','Indonesia','Iraq','Israel','Japan','Jordan','Kazakhstan','Kuwait','Kyrgyzstan','Malaysia',
    'Mongolia','Philippines','Qatar','Saudi Arabia','Singapore','Sri Lanka','Taiwan','Tajikistan','Thailand']

africa = [
    'Angola','Benin','Botswana','Burkina Faso','Burundi','Cameroon','Central African Republic','Egypt','Eswatini',
    'Gabon','Kenya','Lesotho','Mauritania','Mauritius','Morocco','Mozambique','Namibia','Niger','Nigeria',
    'Rwanda','Senegal','Sierra Leone','South Africa','Sudan','Togo','Tunisia','Zambia','Zimbabwe']

latin_america = [
    'Argentina','Barbados','Brazil','Chile','Colombia','Costa Rica','Dominican Republic','Ecuador','El Salvador',
    'Guatemala','Honduras','Jamaica','Mexico','Nicaragua','Panama','Paraguay','Peru','Trinidad and Tobago','Uruguay']

north_america = ['Canada', 'United States']

oceania = ['Australia', 'Fiji', 'New Zealand']

middle_east = ['Bahrain', 'Iraq', 'Israel', 'Jordan', 'Kuwait', 'Qatar', 'Saudi Arabia']

# Regionale dummies
df_model['Europe'] = df_model['country'].isin(europe).astype(int)
df_model['Asia'] = df_model['country'].isin(asia).astype(int)
df_model['Africa'] = df_model['country'].isin(africa).astype(int)
df_model['LatinAmerica'] = df_model['country'].isin(latin_america).astype(int)
df_model['NorthAmerica'] = df_model['country'].isin(north_america).astype(int)
df_model['Oceania'] = df_model['country'].isin(oceania).astype(int)
df_model['MiddleEast'] = df_model['country'].isin(middle_east).astype(int)

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): + kontroller
X2 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + kontroller
X3 = sm.add_constant(df_model[['ODR2002', 'Urban2002']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + kontroller
X4 = sm.add_constant(df_model[['ODR2002','landlocked']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + kontroller
X5 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')

# Model (6): + kontroller
X6 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked','HC2002']])
model6 = sm.OLS(y, X6).fit(cov_type='HC1')

# Model (7): + kontroller
X7 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked','log_GDPpc2002']])
model7 = sm.OLS(y, X7).fit(cov_type='HC1')

# Model (8): + TradeOpen2002
X8 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked','Europe', 'Asia', 'Africa','LatinAmerica', 'NorthAmerica','Oceania', 'MiddleEast']])
model8 = sm.OLS(y, X8).fit(cov_type='HC1')

# Model (9): + TradeOpen2002
X9 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked','HC2002','log_GDPpc2002','Europe', 'Asia', 'Africa','LatinAmerica', 'NorthAmerica','Oceania', 'MiddleEast']])
model9 = sm.OLS(y, X9).fit(cov_type='HC1')

results = summary_col(
    [model1, model2, model3, model4, model5, model6, model7, model8, model9],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)', '(6)', '(7)', '(8)', '(9)'],
    regressor_order=[
        'ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked','HC2002', 'log_GDPpc2002',
        'Europe', 'Asia', 'Africa', 'LatinAmerica',
        'NorthAmerica', 'Oceania', 'MiddleEast'],
    drop_omitted=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)

MODEL:
GrowthTFP = α + β1 ODR2002 + β2 HC2002 + β3 log(GDPpc2002) + β4 Europe + β5 Asia + β6 Africa + β7 LatinAmerica + β8 NorthAmerica + β9 Oceania + β10 MiddleEast + β11 TradeOpen2002 + β12 Urban2002 + β13 landlocked + ε
Antal lande: 91
Antal observationer: 91

                  (1)        (2)        (3)        (4)        (5)        (6)        (7)        (8)       (9)    
----------------------------------------------------------------------------------------------------------------
ODR2002        -0.0091*** -0.0091*** -0.0101*** -0.0094*** -0.0104*** -0.0143*** 0.0008     -0.0175*  -0.0107   
               (0.0031)   (0.0032)   (0.0038)   (0.0032)   (0.0039)   (0.0049)   (0.0042)   (0.0091)  (0.0073)  
TradeOpen2002             -0.0003                          -0.0004    -0.0005    0.0005     -0.0011*  -0.0003   
                          (0.0004)                         (0.0004)   (0.0005)   (0.0005)   (0.0006)  (0.0006)  
Urban2002                            0.0008               

# HC + log(GDP) + regionale dummies + geo variabler

GrowthTFP = α + β1 ODR2002 + β2 HC2002 + β3 log(GDPpc2002) + β4 Europe + β5 Asia + β6 Africa + β7 LatinAmerica + β8 NorthAmerica + β9 Oceania + β10 MiddleEast + β11 TradeOpen2002 + β12 Urban2002 + β13 landlocked + ε

In [164]:
# Klargør datasæt
df_model = Tvaersnit[[
    'country',
    'GrowthTFP',
    'HC2002',
    'ODR2002',
    'GDPpc2002',
    'TradeOpen2002',
    'Urban2002',
    'landlocked'
]].dropna().copy()

# Log-variable
df_model['log_GDPpc2002'] = np.log(df_model['GDPpc2002'])

# Antal lande og observationer
antal_lande = df_model['country'].nunique()
antal_observationer = df_model.shape[0]

# Automatisk print
print("=" * 220)
print("MODEL:")
print("GrowthTFP = α + β1 ODR2002 + β2 HC2002 + β3 log(GDPpc2002) + β4 Europe + β5 Asia + β6 Africa + β7 LatinAmerica + β8 NorthAmerica + β9 Oceania + β10 MiddleEast + β11 TradeOpen2002 + β12 Urban2002 + β13 landlocked + ε")
print(f"Antal lande: {antal_lande}")
print(f"Antal observationer: {antal_observationer}")
print("=" * 220)

# Regioner
europe = [
    'Albania','Austria','Belgium','Bulgaria','Croatia','Cyprus','Czechia','Denmark','Estonia','Finland','France','Germany',
    'Greece','Hungary','Iceland','Ireland','Italy','Latvia','Lithuania','Luxembourg','Malta','Netherlands','Norway',
    'Poland','Portugal','Romania','Serbia','Slovakia','Slovenia','Spain','Sweden','Switzerland','Ukraine','United Kingdom','Armenia']

asia = [
    'Bahrain','China','India','Indonesia','Iraq','Israel','Japan','Jordan','Kazakhstan','Kuwait','Kyrgyzstan','Malaysia',
    'Mongolia','Philippines','Qatar','Saudi Arabia','Singapore','Sri Lanka','Taiwan','Tajikistan','Thailand']

africa = [
    'Angola','Benin','Botswana','Burkina Faso','Burundi','Cameroon','Central African Republic','Egypt','Eswatini',
    'Gabon','Kenya','Lesotho','Mauritania','Mauritius','Morocco','Mozambique','Namibia','Niger','Nigeria',
    'Rwanda','Senegal','Sierra Leone','South Africa','Sudan','Togo','Tunisia','Zambia','Zimbabwe']

latin_america = [
    'Argentina','Barbados','Brazil','Chile','Colombia','Costa Rica','Dominican Republic','Ecuador','El Salvador',
    'Guatemala','Honduras','Jamaica','Mexico','Nicaragua','Panama','Paraguay','Peru','Trinidad and Tobago','Uruguay']

north_america = ['Canada', 'United States']

oceania = ['Australia', 'Fiji', 'New Zealand']

middle_east = ['Bahrain', 'Iraq', 'Israel', 'Jordan', 'Kuwait', 'Qatar', 'Saudi Arabia']

# Regionale dummies
df_model['Europe'] = df_model['country'].isin(europe).astype(int)
df_model['Asia'] = df_model['country'].isin(asia).astype(int)
df_model['Africa'] = df_model['country'].isin(africa).astype(int)
df_model['LatinAmerica'] = df_model['country'].isin(latin_america).astype(int)
df_model['NorthAmerica'] = df_model['country'].isin(north_america).astype(int)
df_model['Oceania'] = df_model['country'].isin(oceania).astype(int)
df_model['MiddleEast'] = df_model['country'].isin(middle_east).astype(int)

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): + kontroller
X2 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + kontroller
X3 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked', 'HC2002']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + kontroller
X4 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked', 'HC2002', 'log_GDPpc2002']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + kontroller
X5 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked','HC2002', 'log_GDPpc2002', 'Europe', 'Asia', 'Africa','LatinAmerica', 'NorthAmerica','Oceania', 'MiddleEast']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')


results = summary_col(
    [model1, model2, model3, model4, model5],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)'],
    regressor_order=[
        'ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked','HC2002', 'log_GDPpc2002',
        'Europe', 'Asia', 'Africa', 'LatinAmerica',
        'NorthAmerica', 'Oceania', 'MiddleEast'],
    drop_omitted=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)


MODEL:
GrowthTFP = α + β1 ODR2002 + β2 HC2002 + β3 log(GDPpc2002) + β4 Europe + β5 Asia + β6 Africa + β7 LatinAmerica + β8 NorthAmerica + β9 Oceania + β10 MiddleEast + β11 TradeOpen2002 + β12 Urban2002 + β13 landlocked + ε
Antal lande: 91
Antal observationer: 91

                  (1)        (2)        (3)        (4)        (5)    
---------------------------------------------------------------------
ODR2002        -0.0091*** -0.0104*** -0.0143*** -0.0059    -0.0107   
               (0.0031)   (0.0039)   (0.0049)   (0.0051)   (0.0073)  
TradeOpen2002             -0.0004    -0.0005    0.0003     -0.0003   
                          (0.0004)   (0.0005)   (0.0005)   (0.0006)  
Urban2002                 0.0009     0.0003     0.0059***  0.0086*** 
                          (0.0016)   (0.0017)   (0.0021)   (0.0030)  
landlocked                -0.0286    -0.0376    -0.0823    -0.0816   
                          (0.0925)   (0.0883)   (0.0799)   (0.0850)  
HC2002                              

# Tabeller

In [165]:
# Klargør datasæt
df_model = Tvaersnit[[
    'country',
    'GrowthTFP',
    'HC2002',
    'ODR2002',
    'GDPpc2002',
    'TradeOpen2002',
    'Urban2002',
    'landlocked'
]].dropna().copy()

# Log-variable
df_model['log_GDPpc2002'] = np.log(df_model['GDPpc2002'])

# Antal lande og observationer
antal_lande = df_model['country'].nunique()
antal_observationer = df_model.shape[0]

# Automatisk print
print("=" * 220)
print("MODEL:")
print("GrowthTFP = α + β1 ODR2002 + β2 HC2002 + β3 log(GDPpc2002) + β4 Europe + β5 Asia + β6 Africa + β7 LatinAmerica + β8 NorthAmerica + β9 Oceania + β10 MiddleEast + β11 TradeOpen2002 + β12 Urban2002 + β13 landlocked + ε")
print(f"Antal lande: {antal_lande}")
print(f"Antal observationer: {antal_observationer}")
print("=" * 220)

# Regioner
europe = [
    'Albania','Austria','Belgium','Bulgaria','Croatia','Cyprus','Czechia','Denmark','Estonia','Finland','France','Germany',
    'Greece','Hungary','Iceland','Ireland','Italy','Latvia','Lithuania','Luxembourg','Malta','Netherlands','Norway',
    'Poland','Portugal','Romania','Serbia','Slovakia','Slovenia','Spain','Sweden','Switzerland','Ukraine','United Kingdom','Armenia']

asia = [
    'Bahrain','China','India','Indonesia','Iraq','Israel','Japan','Jordan','Kazakhstan','Kuwait','Kyrgyzstan','Malaysia',
    'Mongolia','Philippines','Qatar','Saudi Arabia','Singapore','Sri Lanka','Taiwan','Tajikistan','Thailand']

africa = [
    'Angola','Benin','Botswana','Burkina Faso','Burundi','Cameroon','Central African Republic','Egypt','Eswatini',
    'Gabon','Kenya','Lesotho','Mauritania','Mauritius','Morocco','Mozambique','Namibia','Niger','Nigeria',
    'Rwanda','Senegal','Sierra Leone','South Africa','Sudan','Togo','Tunisia','Zambia','Zimbabwe']

latin_america = [
    'Argentina','Barbados','Brazil','Chile','Colombia','Costa Rica','Dominican Republic','Ecuador','El Salvador',
    'Guatemala','Honduras','Jamaica','Mexico','Nicaragua','Panama','Paraguay','Peru','Trinidad and Tobago','Uruguay']

north_america = ['Canada', 'United States']

oceania = ['Australia', 'Fiji', 'New Zealand']

middle_east = ['Bahrain', 'Iraq', 'Israel', 'Jordan', 'Kuwait', 'Qatar', 'Saudi Arabia']

# Regionale dummies
df_model['Europe'] = df_model['country'].isin(europe).astype(int)
df_model['Asia'] = df_model['country'].isin(asia).astype(int)
df_model['Africa'] = df_model['country'].isin(africa).astype(int)
df_model['LatinAmerica'] = df_model['country'].isin(latin_america).astype(int)
df_model['NorthAmerica'] = df_model['country'].isin(north_america).astype(int)
df_model['Oceania'] = df_model['country'].isin(oceania).astype(int)
df_model['MiddleEast'] = df_model['country'].isin(middle_east).astype(int)

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): + kontroller
X2 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + kontroller
X3 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked', 'HC2002']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + kontroller
X4 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked','HC2002', 'Europe', 'Asia', 'Africa','LatinAmerica', 'NorthAmerica','Oceania', 'MiddleEast']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')


results = summary_col(
    [model1, model2, model3, model4],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)'],
    regressor_order=[
        'ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked','HC2002',
        'Europe', 'Asia', 'Africa', 'LatinAmerica',
        'NorthAmerica', 'Oceania', 'MiddleEast'],
    drop_omitted=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)

MODEL:
GrowthTFP = α + β1 ODR2002 + β2 HC2002 + β3 log(GDPpc2002) + β4 Europe + β5 Asia + β6 Africa + β7 LatinAmerica + β8 NorthAmerica + β9 Oceania + β10 MiddleEast + β11 TradeOpen2002 + β12 Urban2002 + β13 landlocked + ε
Antal lande: 91
Antal observationer: 91

                  (1)        (2)        (3)        (4)   
---------------------------------------------------------
ODR2002        -0.0091*** -0.0104*** -0.0143*** -0.0196**
               (0.0031)   (0.0039)   (0.0049)   (0.0097) 
TradeOpen2002             -0.0004    -0.0005    -0.0011* 
                          (0.0004)   (0.0005)   (0.0006) 
Urban2002                 0.0009     0.0003     0.0033   
                          (0.0016)   (0.0017)   (0.0027) 
landlocked                -0.0286    -0.0376    -0.0342  
                          (0.0925)   (0.0883)   (0.0896) 
HC2002                               0.0739     0.0606   
                                     (0.0840)   (0.0943) 
Europe                                  

In [166]:
# Klargør datasæt
df_model = Tvaersnit[[
    'country',
    'GrowthTFP',
    'HC2002',
    'ODR2002',
    'GDPpc2002',
    'TradeOpen2002',
    'Urban2002',
    'landlocked',
    'TFP2002'
]].dropna().copy()

# Log-variable
df_model['log_GDPpc2002'] = np.log(df_model['GDPpc2002'])

# Antal lande og observationer
antal_lande = df_model['country'].nunique()
antal_observationer = df_model.shape[0]

# Automatisk print
print("=" * 220)
print("MODEL:")
print("GrowthTFP = α + β1 ODR2002 + β2 HC2002 + β3 log(GDPpc2002) + β4 Europe + β5 Asia + β6 Africa + β7 LatinAmerica + β8 NorthAmerica + β9 Oceania + β10 MiddleEast + β11 TradeOpen2002 + β12 Urban2002 + β13 landlocked + ε")
print(f"Antal lande: {antal_lande}")
print(f"Antal observationer: {antal_observationer}")
print("=" * 220)

# Regioner
europe = [
    'Albania','Austria','Belgium','Bulgaria','Croatia','Cyprus','Czechia','Denmark','Estonia','Finland','France','Germany',
    'Greece','Hungary','Iceland','Ireland','Italy','Latvia','Lithuania','Luxembourg','Malta','Netherlands','Norway',
    'Poland','Portugal','Romania','Serbia','Slovakia','Slovenia','Spain','Sweden','Switzerland','Ukraine','United Kingdom','Armenia']

asia = [
    'Bahrain','China','India','Indonesia','Iraq','Israel','Japan','Jordan','Kazakhstan','Kuwait','Kyrgyzstan','Malaysia',
    'Mongolia','Philippines','Qatar','Saudi Arabia','Singapore','Sri Lanka','Taiwan','Tajikistan','Thailand']

africa = [
    'Angola','Benin','Botswana','Burkina Faso','Burundi','Cameroon','Central African Republic','Egypt','Eswatini',
    'Gabon','Kenya','Lesotho','Mauritania','Mauritius','Morocco','Mozambique','Namibia','Niger','Nigeria',
    'Rwanda','Senegal','Sierra Leone','South Africa','Sudan','Togo','Tunisia','Zambia','Zimbabwe']

latin_america = [
    'Argentina','Barbados','Brazil','Chile','Colombia','Costa Rica','Dominican Republic','Ecuador','El Salvador',
    'Guatemala','Honduras','Jamaica','Mexico','Nicaragua','Panama','Paraguay','Peru','Trinidad and Tobago','Uruguay']

north_america = ['Canada', 'United States']

oceania = ['Australia', 'Fiji', 'New Zealand']

middle_east = ['Bahrain', 'Iraq', 'Israel', 'Jordan', 'Kuwait', 'Qatar', 'Saudi Arabia']

# Regionale dummies
df_model['Europe'] = df_model['country'].isin(europe).astype(int)
df_model['Asia'] = df_model['country'].isin(asia).astype(int)
df_model['Africa'] = df_model['country'].isin(africa).astype(int)
df_model['LatinAmerica'] = df_model['country'].isin(latin_america).astype(int)
df_model['NorthAmerica'] = df_model['country'].isin(north_america).astype(int)
df_model['Oceania'] = df_model['country'].isin(oceania).astype(int)
df_model['MiddleEast'] = df_model['country'].isin(middle_east).astype(int)

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): + kontroller
X2 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + kontroller
X3 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked', 'HC2002']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + kontroller
X4 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked','HC2002', 'Europe', 'Asia', 'Africa','LatinAmerica', 'NorthAmerica','Oceania', 'MiddleEast']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + kontroller
X5 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked','HC2002', 'Europe', 'Asia', 'Africa','LatinAmerica', 'NorthAmerica','Oceania', 'MiddleEast', 'log_GDPpc2002']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')

# Model (6): + kontroller
X6 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked','HC2002', 'Europe', 'Asia', 'Africa','LatinAmerica', 'NorthAmerica','Oceania', 'MiddleEast', 'TFP2002']])
model6 = sm.OLS(y, X6).fit(cov_type='HC1')

results = summary_col(
    [model1, model2, model3, model4, model5, model6],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)', '(6)'],
    regressor_order=[
        'ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked','HC2002',
        'Europe', 'Asia', 'Africa', 'LatinAmerica',
        'NorthAmerica', 'Oceania', 'MiddleEast', 'log_GDPpc2002', 'TFP2002'],
    drop_omitted=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)

MODEL:
GrowthTFP = α + β1 ODR2002 + β2 HC2002 + β3 log(GDPpc2002) + β4 Europe + β5 Asia + β6 Africa + β7 LatinAmerica + β8 NorthAmerica + β9 Oceania + β10 MiddleEast + β11 TradeOpen2002 + β12 Urban2002 + β13 landlocked + ε
Antal lande: 91
Antal observationer: 91

                  (1)        (2)        (3)        (4)       (5)        (6)    
-------------------------------------------------------------------------------
ODR2002        -0.0091*** -0.0104*** -0.0143*** -0.0196** -0.0107    -0.0030   
               (0.0031)   (0.0039)   (0.0049)   (0.0097)  (0.0073)   (0.0073)  
TradeOpen2002             -0.0004    -0.0005    -0.0011*  -0.0003    0.0002    
                          (0.0004)   (0.0005)   (0.0006)  (0.0006)   (0.0006)  
Urban2002                 0.0009     0.0003     0.0033    0.0086***  0.0047**  
                          (0.0016)   (0.0017)   (0.0027)  (0.0030)   (0.0021)  
landlocked                -0.0286    -0.0376    -0.0342   -0.0816    -0.0993   
                

In [170]:
# Klargør datasæt
df_model = Tvaersnit[[
    'country',
    'GrowthTFP',
    'HC2002',
    'ODR2002',
    'GDPpc2002',
    'TradeOpen2002',
    'Urban2002',
    'landlocked',
    'TFP2002'
]].dropna().copy()

# Log-variable
df_model['log_GDPpc2002'] = np.log(df_model['GDPpc2002'])

# Antal lande og observationer
antal_lande = df_model['country'].nunique()
antal_observationer = df_model.shape[0]

# Automatisk print
print("=" * 220)
print("MODEL:")
print("GrowthTFP = α + β1 ODR2002 + β2 HC2002 + β3 log(GDPpc2002) + β4 Europe + β5 Asia + β6 Africa + β7 LatinAmerica + β8 NorthAmerica + β9 Oceania + β10 MiddleEast + β11 TradeOpen2002 + β12 Urban2002 + β13 landlocked + ε")
print(f"Antal lande: {antal_lande}")
print(f"Antal observationer: {antal_observationer}")
print("=" * 220)

# Regioner
europe = [
    'Albania','Austria','Belgium','Bulgaria','Croatia','Cyprus','Czechia','Denmark','Estonia','Finland','France','Germany',
    'Greece','Hungary','Iceland','Ireland','Italy','Latvia','Lithuania','Luxembourg','Malta','Netherlands','Norway',
    'Poland','Portugal','Romania','Serbia','Slovakia','Slovenia','Spain','Sweden','Switzerland','Ukraine','United Kingdom','Armenia']

asia = [
    'Bahrain','China','India','Indonesia','Iraq','Israel','Japan','Jordan','Kazakhstan','Kuwait','Kyrgyzstan','Malaysia',
    'Mongolia','Philippines','Qatar','Saudi Arabia','Singapore','Sri Lanka','Taiwan','Tajikistan','Thailand']

africa = [
    'Angola','Benin','Botswana','Burkina Faso','Burundi','Cameroon','Central African Republic','Egypt','Eswatini',
    'Gabon','Kenya','Lesotho','Mauritania','Mauritius','Morocco','Mozambique','Namibia','Niger','Nigeria',
    'Rwanda','Senegal','Sierra Leone','South Africa','Sudan','Togo','Tunisia','Zambia','Zimbabwe']

latin_america = [
    'Argentina','Barbados','Brazil','Chile','Colombia','Costa Rica','Dominican Republic','Ecuador','El Salvador',
    'Guatemala','Honduras','Jamaica','Mexico','Nicaragua','Panama','Paraguay','Peru','Trinidad and Tobago','Uruguay']

north_america = ['Canada', 'United States']

oceania = ['Australia', 'Fiji', 'New Zealand']

middle_east = ['Bahrain', 'Iraq', 'Israel', 'Jordan', 'Kuwait', 'Qatar', 'Saudi Arabia']

# Regionale dummies
df_model['Europe'] = df_model['country'].isin(europe).astype(int)
df_model['Asia'] = df_model['country'].isin(asia).astype(int)
df_model['Africa'] = df_model['country'].isin(africa).astype(int)
df_model['LatinAmerica'] = df_model['country'].isin(latin_america).astype(int)
df_model['NorthAmerica'] = df_model['country'].isin(north_america).astype(int)
df_model['Oceania'] = df_model['country'].isin(oceania).astype(int)
df_model['MiddleEast'] = df_model['country'].isin(middle_east).astype(int)

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): + kontroller
X2 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + kontroller
X3 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked', 'HC2002']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + kontroller
X4 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked','HC2002', 'Europe', 'Asia', 'Africa','LatinAmerica', 'NorthAmerica','Oceania', 'MiddleEast']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + kontroller
X5 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked', 'TFP2002']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')

# Model (6): + kontroller
X6 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked', 'log_GDPpc2002']])
model6 = sm.OLS(y, X6).fit(cov_type='HC1')

results = summary_col(
    [model1, model2, model3, model4, model5, model6],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)', '(6)'],
    regressor_order=[
        'ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked','HC2002',
        'Europe', 'Asia', 'Africa', 'LatinAmerica',
        'NorthAmerica', 'Oceania', 'MiddleEast', 'TFP2002', 'log_GDPpc2002'],
    drop_omitted=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)

MODEL:
GrowthTFP = α + β1 ODR2002 + β2 HC2002 + β3 log(GDPpc2002) + β4 Europe + β5 Asia + β6 Africa + β7 LatinAmerica + β8 NorthAmerica + β9 Oceania + β10 MiddleEast + β11 TradeOpen2002 + β12 Urban2002 + β13 landlocked + ε
Antal lande: 91
Antal observationer: 91

                  (1)        (2)        (3)        (4)       (5)        (6)    
-------------------------------------------------------------------------------
ODR2002        -0.0091*** -0.0104*** -0.0143*** -0.0196** 0.0072*    0.0008    
               (0.0031)   (0.0039)   (0.0049)   (0.0097)  (0.0039)   (0.0042)  
TradeOpen2002             -0.0004    -0.0005    -0.0011*  0.0005     0.0005    
                          (0.0004)   (0.0005)   (0.0006)  (0.0004)   (0.0005)  
Urban2002                 0.0009     0.0003     0.0033    0.0036***  0.0063*** 
                          (0.0016)   (0.0017)   (0.0027)  (0.0014)   (0.0022)  
landlocked                -0.0286    -0.0376    -0.0342   -0.0690    -0.0575   
                